# BAMoE — Bias-Aware Mixture of Experts for Time Series Forecasting

Run all five experiments end-to-end:
1. **Preliminary** — single-bias Transformer comparison
2. **Main results** — BAMoE vs. baselines
3. **Ablation** — expert diversity, routing mechanism, K-sweep
4. **Interpretability** — routing dynamics and attention patterns
5. **Efficiency** — parameter count vs. performance

> **Before running:** make sure the repo is public (or you have a token), and that `Runtime → Change runtime type` is set to **GPU**.
>
> Results are downloaded automatically after each experiment completes.

## 0 · Configuration
Edit the values in this cell before running anything else.

In [1]:
# ── Repository ────────────────────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/hingma/BAMoE.git"
BRANCH      = "main"

# ── Paths ─────────────────────────────────────────────────────────────────────
# Set USE_DRIVE=True to persist checkpoints/results across Colab sessions.
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/BAMoE"   # ignored when USE_DRIVE=False

# ── Default model hyperparameters ─────────────────────────────────────────────
CFG = dict(
    seq_len   = 336,
    d_model   = 128,
    n_heads   = 8,
    n_layers  = 3,
    d_ff      = 256,
    dropout   = 0.1,
    patch_len = 16,
    stride    = 8,
    # BAMoE-specific
    expert_types      = "causal,local,periodic,global",
    top_k             = 2,
    routing           = "learned_sparse",
    load_balance_coef = 0.01,
    local_window      = 3,
    periodic_period   = 12,
    # Training
    batch_size    = 128,
    learning_rate = 1e-4,
    train_epochs  = 20,
    patience      = 5,
    weight_decay  = 1e-4,
    lradj         = "cosine",
    # Data
    features    = "M",
    target      = "OT",
    num_workers = 2,
)

## 1 · Environment setup

In [2]:
import subprocess, sys, os

# Reduce CUDA memory fragmentation across many sequential runs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

IN_COLAB = "google.colab" in sys.modules

# GPU info
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                    "--format=csv,noheader"], capture_output=True, text=True)
print("GPU :", r.stdout.strip() if r.returncode == 0 else "none — running on CPU")
print("Python:", sys.version.split()[0])

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("Drive mounted ->", DRIVE_DIR)


GPU : Tesla T4, 15360 MiB
Python: 3.12.13


## 2 · Install dependencies

In [3]:
!pip install torch numpy pandas scikit-learn matplotlib seaborn scipy tqdm --quiet

## 3 · Clone repository

In [4]:
WORK_DIR = DRIVE_DIR if (IN_COLAB and USE_DRIVE) else "/content/BAMoE"

if os.path.isdir(os.path.join(WORK_DIR, ".git")):
    print("Repo already cloned — pulling latest changes...")
    !git -C {WORK_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {GITHUB_REPO} {WORK_DIR}

%cd {WORK_DIR}
print("Working directory:", os.getcwd())

Cloning into '/content/BAMoE'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 40 (delta 2), reused 40 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 27.42 KiB | 9.14 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/BAMoE
Working directory: /content/BAMoE


## 4 · Download datasets

In [5]:
!bash scripts/download_data.sh ./data

=== ETT datasets ===
  downloading: ETTh1.csv
######################################################################## 100.0%
  downloading: ETTh2.csv
######################################################################## 100.0%
  downloading: ETTm1.csv
######################################################################## 100.0%
  downloading: ETTm2.csv
######################################################################## 100.0%
=== Weather ===
  downloading: weather.csv
######################################################################## 100.0%
=== Traffic ===
  downloading: traffic.csv
######################################################################## 100.0%
=== Electricity ===
  downloading: electricity.csv
######################################################################## 100.0%##O#-#                                                                        
=== Exchange Rate ===
  downloading: exchange_rate.csv
#################################################

## 5 · Helpers

In [6]:
import argparse, shutil, gc, random
import numpy as np
import torch

# ── Per-dataset overrides for large-feature datasets ──────────────────────────
# Traffic (862 cols) and Electricity (321 cols) produce very wide
# channel-independent batches; small batch + no DataLoader workers is safer.
DATASET_OVERRIDES = {
    "Traffic":     dict(batch_size=4,  num_workers=0),
    "Electricity": dict(batch_size=16, num_workers=0),
}

# ── Experiment runner ──────────────────────────────────────────────────────────

def make_args(overrides: dict) -> argparse.Namespace:
    """Merge CFG defaults with per-experiment overrides into an args Namespace."""
    d = dict(
        root_path   = "./data",
        data_path   = "",
        checkpoints = "./checkpoints",
        results     = "./results",
        exp_name    = "",
        resume      = True,
        use_gpu     = 1,
        use_amp     = True,
        gpu         = 0,
        model       = "BAMoE",
        bias_type   = "global",
        **CFG,
    )
    # Apply dataset-specific defaults first, then caller overrides on top
    data_key = overrides.get("data", d["data"])
    d.update(DATASET_OVERRIDES.get(data_key, {}))
    d.update(overrides)
    if not d["exp_name"]:
        if d["model"] == "SingleBias":
            tag = f"SingleBias_{d['bias_type']}"
        else:
            k = len(d["expert_types"].split(","))
            tag = f"BAMoE_K{k}_{d['routing']}"
        d["exp_name"] = f"{tag}_{d['data']}_sl{d['seq_len']}_pl{d['pred_len']}"
    return argparse.Namespace(**d)


def set_seed(seed=2024):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def _free_gpu(exp):
    """Reliably release all GPU memory held by an ExpForecast instance.

    Moving the model to CPU frees GPU tensors immediately without waiting
    for Python's GC — the key difference from just 'del exp'.
    """
    try:
        exp.model.cpu()
    except Exception:
        pass
    del exp
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        free_gb = torch.cuda.mem_get_info()[0] / 1e9
        print(f"  GPU free after cleanup: {free_gb:.1f} GB")


def run_one(overrides: dict):
    """Train + test one configuration, then free all GPU memory before returning."""
    from exp.exp_forecast import ExpForecast
    set_seed()
    args = make_args(overrides)
    print(f"\n{'='*60}\n{args.exp_name}  "
          f"[bs={args.batch_size}, workers={args.num_workers}, amp={args.use_amp}]")
    exp = None
    try:
        exp = ExpForecast(args)
        exp.train()
        result = exp.test()
    except torch.cuda.OutOfMemoryError:
        print("  OOM — cleaning up and skipping this run.")
        result = (float('nan'),) * 4
    finally:
        if exp is not None:
            _free_gpu(exp)
    return result


# ── Download helper ────────────────────────────────────────────────────────────

def download_results(label: str):
    """
    Zip the current results/ folder and trigger a browser download.
    Each call produces bamoe_<label>.zip so files do not overwrite each other.
    """
    zip_name = f"bamoe_{label}"
    shutil.make_archive(zip_name, "zip", "results")
    size_mb = os.path.getsize(f"{zip_name}.zip") / 1e6
    print(f"\n📦  {zip_name}.zip  ({size_mb:.1f} MB) — ", end="")
    if IN_COLAB:
        from google.colab import files
        files.download(f"{zip_name}.zip")
        print("download started.")
    else:
        print(f"saved to {os.path.abspath(zip_name + '.zip')}")


print("Helpers ready.")


Helpers ready.


---
## Experiment 1 — Preliminary: Temporal Inductive Bias Analysis

Trains four single-bias Transformers (global / causal / local / periodic) across
datasets and horizons to show that no single bias dominates — the key motivation
for BAMoE.

Adjust `EXP1_DATASETS` / `EXP1_PRED_LENS` to run a faster subset.

In [ ]:
EXP1_BIAS_TYPES = ["global", "causal", "local", "periodic"]
EXP1_DATASETS   = ["ETTh1", "ETTh2", "Weather", "Traffic"]  # extend with ETTm1/m2, Electricity, Exchange
EXP1_PRED_LENS  = [96, 192, 336, 720]

for bias in EXP1_BIAS_TYPES:
    for data in EXP1_DATASETS:
        for pl in EXP1_PRED_LENS:
            run_one(dict(model="SingleBias", bias_type=bias, data=data, pred_len=pl))

download_results("exp1_preliminary")


SingleBias_global_ETTh1_sl336_pl96
Model: SingleBias  |  Parameters: 902,624


Epoch   1 | train=0.4861 val=0.7908 | lr=9.94e-05 | 6.5s


Epoch   2 | train=0.3816 val=0.7508 | lr=9.76e-05 | 5.6s


Epoch   3 | train=0.3646 val=0.7321 | lr=9.46e-05 | 5.7s


Epoch   4 | train=0.3548 val=0.7247 | lr=9.05e-05 | 5.7s


...

Epoch   5 | train=0.4106 val=0.3807 | lr=8.55e-05 | 80.5s


Epoch   6 | train=0.4078 val=0.3813 | lr=7.96e-05 | 78.7s


Epoch   7 | train=0.4044 val=0.3799 | lr=7.30e-05 | 78.6s


Epoch   8 | train=0.4027 val=0.3796 | lr=6.58e-05 | 78.5s


Epoch 9/20:  63%|██████▎   | 180/284 [00:47<00:28,  3.69it/s]

---
## Experiment 2 — Main BAMoE Results

In [ ]:
EXP2_DATASETS  = ["ETTh1", "ETTh2", "ETTm1", "ETTm2",
                  "Weather", "Traffic", "Electricity", "Exchange"]
EXP2_PRED_LENS = [96, 192, 336, 720]

for data in EXP2_DATASETS:
    for pl in EXP2_PRED_LENS:
        run_one(dict(model="BAMoE", data=data, pred_len=pl))

download_results("exp2_main")

---
## Experiment 3 — Ablation Studies
### 3a · Expert diversity

In [ ]:
EXP3_DATASETS  = ["ETTh1", "Weather", "Traffic"]
EXP3_PRED_LENS = [96, 192, 336, 720]

DIVERSITY_VARIANTS = {
    "homo_causal"   : "causal,causal,causal,causal",
    "homo_local"    : "local,local,local,local",
    "homo_periodic" : "periodic,periodic,periodic,periodic",
    "homo_global"   : "global,global,global,global",
    "K1"            : "causal",
    "hetero"        : "causal,local,periodic,global",
}

for tag, etypes in DIVERSITY_VARIANTS.items():
    for data in EXP3_DATASETS:
        for pl in EXP3_PRED_LENS:
            run_one(dict(model="BAMoE", expert_types=etypes, data=data, pred_len=pl,
                         exp_name=f"ablation_diversity_{tag}_{data}_pl{pl}"))

download_results("exp3a_diversity")

### 3b · Routing mechanism

In [ ]:
ROUTING_VARIANTS = ["uniform", "random", "top1", "learned_sparse", "dense"]

for routing in ROUTING_VARIANTS:
    k = 1 if routing == "top1" else 2
    for data in EXP3_DATASETS:
        for pl in EXP3_PRED_LENS:
            run_one(dict(model="BAMoE", routing=routing, top_k=k,
                         data=data, pred_len=pl,
                         exp_name=f"ablation_routing_{routing}_{data}_pl{pl}"))

download_results("exp3b_routing")

### 3c · Number of experts K

In [ ]:
K_VARIANTS = {
    "K2" : "causal,global",
    "K3" : "causal,local,periodic",
    "K4" : "causal,local,periodic,global",
    "K6" : "causal,local,periodic,global,causal,local",
    "K8" : "causal,local,periodic,global,causal,local,periodic,global",
}

for tag, etypes in K_VARIANTS.items():
    for data in EXP3_DATASETS:
        for pl in EXP3_PRED_LENS:
            run_one(dict(model="BAMoE", expert_types=etypes,
                         data=data, pred_len=pl,
                         exp_name=f"ablation_Ksweep_{tag}_{data}_pl{pl}"))

download_results("exp3c_k_sweep")

---
## Experiment 4 — Interpretability Analysis
Requires completed Experiment 2 checkpoints.

In [ ]:
from exp.exp_interpretability import ExpInterpretability

EXP4_DATASETS  = ["ETTh1", "ETTh2", "Weather", "Traffic"]
EXP4_PRED_LENS = [96, 192, 336, 720]

for data in EXP4_DATASETS:
    for pl in EXP4_PRED_LENS:
        args = make_args(dict(model="BAMoE", data=data, pred_len=pl))
        ExpInterpretability(args).run()

download_results("exp4_interpretability")

---
## Results — Summary Table

In [ ]:
import pandas as pd

df = pd.read_csv("results/summary.csv")
pivot = (
    df.groupby(["model", "data", "pred_len"])[["mse", "mae", "crps", "mase"]]
    .mean()
    .round(4)
)
pd.set_option("display.max_rows", 200)
pivot

In [ ]:
import matplotlib.pyplot as plt

avg = df.groupby("model")["mse"].mean().sort_values()
fig, ax = plt.subplots(figsize=(max(6, len(avg) * 0.8), 4))
avg.plot.bar(ax=ax, color="steelblue", edgecolor="white")
ax.set_ylabel("Average MSE")
ax.set_title("Average MSE per model (all datasets × horizons)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("results/avg_mse_bar.pdf", dpi=150)
plt.show()

download_results("final_all")